# Expert analysis

In [18]:
# first we load the data.
# All data is provided in a single excel report sheet which is the single source of trust here
from pathlib import Path
import pandas as pd

ROOT = Path("/local/joeldag/repos/moe-study")
REPORT = ROOT / "outputs/pipeline_runs/moe_study_expert_sweep_pipeline_report/expert_pipeline_report.xlsx"
xls = pd.ExcelFile(REPORT)
raw = pd.read_excel(REPORT, sheet_name="benchmark_raw")
source = pd.read_excel(REPORT, sheet_name="source_manifest")
audit = pd.read_excel(REPORT, sheet_name="audit_status")

primary = raw[
  ((raw.benchmark == "Belebele") & (raw.metric == "acc_norm")) |
  ((raw.benchmark == "FLORES") & (raw.metric.astype(str).str.startswith("bits_per_byte "))) |
  ((raw.benchmark == "SIB200") & (raw.metric == "acc")) |
  ((raw.benchmark == "Taxi1500") & (raw.metric == "acc"))
].copy() # primary metric we use in our analysis
primary = primary[~primary["is_baseline"].astype(str).str.lower().isin(["true", "1", "yes"])] # remove baseline
primary.head()

,run_id,kind,benchmark,task,language,resource_availability,metric,better,value,is_baseline
0,adamoe_1_r2_e16_k1,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.600000,False
1,adamoe_1_r2_e16_k2,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.588889,False
2,adamoe_1_r2_e16_k4,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.594444,False
3,adamoe_1_r2_e16_k8,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.597778,False
4,adamoe_1_r2_e2_k1,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.577778,False


In [19]:
# here we prepare our df for analysis and normalize the scores
import re
import numpy as np

# helper, extracts the expert count from a run_id, e.g. moelora_r2_e16_k8 -> 16, hydralora_r2_heads32 -> 32 mtllora_r2_nup8 -> 8, cola_r2_a3_b4 -> 4
def parse_count(run_id):
  for pat in [r"_e(\d+)(?:_|$)", r"_heads(\d+)$", r"_nup(\d+)$", r"_b(\d+)$"]:
      m = re.search(pat, str(run_id))
      if m:
          return int(m.group(1))
  return np.nan

metrics = primary.copy()
len(metrics)
metrics["direction"] = metrics.metric.astype(str).str.extract(r"(en->x|x->en)", expand=False).fillna("") # extract metrics direction
metrics["score_family"] = np.where(metrics.benchmark.eq("FLORES"), "FLORES " + metrics.direction, metrics.benchmark) #
metrics["component_count"] = metrics.run_id.map(parse_count) # e.g. for 8 experts its 8
metrics["top_k"] = metrics.run_id.astype(str).str.extract(r"_k(\d+)(?:_|$)", expand=False).astype(float)
metrics["axis_label"] = metrics.kind.map({"hydralora": "heads", "mtllora": "up_projections", "cola": "cola_B"}).fillna("experts")
metrics["value"] = pd.to_numeric(metrics.value, errors="coerce") # ensure numeric values, NaN else

# we normalize metrics using min - max normalization. within each benchmark family (belebele, flores, taxi, sib) each result gets a value between 0-1
g = metrics.groupby(["score_family", "task", "metric"]).value
lo, hi = g.transform("min"), g.transform("max")
metrics["normalized_score"] = np.where(metrics.better.eq("lower"), (hi - metrics.value) / (hi - lo), (metrics.value - lo) / (hi - lo))
metrics.loc[(hi - lo).eq(0), "normalized_score"] = 1.0 # edge case if all have same value

metrics = metrics.dropna(subset=["value", "component_count"]).copy()
metrics["component_count"] = metrics.component_count.astype(int)
metrics.head()

,run_id,kind,benchmark,task,language,resource_availability,metric,better,value,is_baseline,direction,score_family,component_count,top_k,axis_label,normalized_score
0,adamoe_1_r2_e16_k1,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.600000,False,,Belebele,16,1.0,experts,0.946779
1,adamoe_1_r2_e16_k2,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.588889,False,,Belebele,16,2.0,experts,0.918768
2,adamoe_1_r2_e16_k4,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.594444,False,,Belebele,16,4.0,experts,0.932773
3,adamoe_1_r2_e16_k8,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.597778,False,,Belebele,16,8.0,experts,0.941176
4,adamoe_1_r2_e2_k1,mola,Belebele,belebele_arb_Arab,arb_Arab,HIGH,acc_norm,higher,0.577778,False,,Belebele,2,1.0,experts,0.890756


In [20]:
# we have 68 training runs * (9xbelebele metrics + 18 FLORES + 9 SIB + 6 TAXI) = 2856 metrics we should have for all approaches x all datasets x all languages
# our df should have that many entries with each entry having the exact metric results as well as the normalize score and metadata of the run
len(metrics)

2856

In [21]:
# last sanity checks
print("rows:", len(metrics))
print("runs:", metrics.run_id.nunique())
print("score families:", metrics.score_family.unique())

# should be 0 missing after prep
print(metrics[["value", "normalized_score", "component_count"]].isna().sum())

# coverage per score family
metrics.groupby("score_family").agg(
  rows=("run_id", "count"),
  runs=("run_id", "nunique"),
  tasks=("task", "nunique"),
  languages=("language", "nunique"))

rows: 2856
runs: 68
score families: ['Belebele' 'FLORES en->x' 'FLORES x->en' 'SIB200' 'Taxi1500']
value               0
normalized_score    0
component_count     0
dtype: int64


,rows,runs,tasks,languages
score_family,,,,
Belebele,612,68,9,9
FLORES en->x,612,68,9,9
FLORES x->en,612,68,9,9
SIB200,612,68,9,9
Taxi1500,408,68,6,6


# Which expert count wins per benchmark and is the result significant?

In [22]:
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests
import pandas as pd
import numpy as np


# We use Friedmann and Wilconxon signed rank tests + Holm correction here for confirmatory post-hoc comparisons.
# holm is used because our goal is to make conservative winner claims while controlling the family-wise error rate: the chance of making at least one false claim.
#
# In contrast, pipeline_tools/reporting/significance.py uses Benjamini-Hochberg FDR correction, which is better suited for exploratory screening across many tests where we want to control the expected share of false discoveries.
# https://en.wikipedia.org/wiki/Holm%E2%80%93Bonferroni_method

COUNTS = [2, 4, 8, 16, 32]

def holm(pvals):
  return multipletests(pd.Series(pvals, dtype=float), method="holm")[1]

def count_pivot(df, group_cols):
  return (
      df.groupby(group_cols + ["component_count"])["normalized_score"]
      .mean()
      .unstack("component_count")
      .reindex(columns=COUNTS)
      .dropna()
  )

def friedman_count_overall(df, label):
  pivot = count_pivot(df, ["score_family", "task", "language", "metric"])

  stat, p = friedmanchisquare(*[pivot[c] for c in pivot.columns])
  avg_ranks = pivot.rank(axis=1, ascending=False).mean().sort_values()
  top3 = avg_ranks.index[:3].tolist()

  return {
      "analysis": label,
      "pivot": pivot,
      "top3_counts": top3,
      "friedman_p_raw": p,
      "kendalls_w": stat / (len(pivot) * (pivot.shape[1] - 1)),
      **{f"avg_rank_{c}": avg_ranks.get(c, np.nan) for c in COUNTS},
  }

def wilcoxon_adjacent_top3(result):
  pivot = result["pivot"]
  top3 = result["top3_counts"]

  rows = []
  # we compare if 1st and 2nd best and 2nd and 3rd best are significant using Wilconxon signed rank
  for a, b in [(top3[0], top3[1]), (top3[1], top3[2])]:
      w = wilcoxon(pivot[a], pivot[b], zero_method="pratt", alternative="greater")
      rows.append({
          "analysis": result["analysis"],
          "a": a,
          "b": b,
          "mean_a": pivot[a].mean(),
          "mean_b": pivot[b].mean(),
          "median_delta_a_minus_b": (pivot[a] - pivot[b]).median(),
          "n_pairs": len(pivot),
          "p_raw": w.pvalue,
      })

  out = pd.DataFrame(rows)
  out["p_holm"] = holm(out["p_raw"])
  out["significant"] = out["p_holm"] < 0.05
  return out

def friedman_count_by_benchmark(df, label):
  rows = []

  # here we analyze per benchmark family and use fridmann test to derive which expert count performs best per family
  for score_family, g in df.groupby("score_family"):
      pivot = count_pivot(g, ["task", "language", "metric"])

      stat, p = friedmanchisquare(*[pivot[c] for c in pivot.columns])
      avg_ranks = pivot.rank(axis=1, ascending=False).mean().sort_values()

      rows.append({
          "analysis": label,
          "score_family": score_family,
          "winner_count": avg_ranks.index[0],
          "n_units": len(pivot),
          "p_raw": p,
          "kendalls_w": stat / (len(pivot) * (pivot.shape[1] - 1)),
          **{f"avg_rank_{c}": avg_ranks.get(c, np.nan) for c in COUNTS},
      })

  out = pd.DataFrame(rows)
  out["p_holm"] = holm(out["p_raw"])
  out["significant"] = out["p_holm"] < 0.05
  return out

# Overall: top 3 counts across all benchmark families
# we check with and without cola since cola seems to skew resutls for higher expert count
overall_all = friedman_count_overall(metrics, "with_cola")
overall_no_cola = friedman_count_overall(metrics[metrics.kind.ne("cola")], "no_cola")

overall_summary = pd.DataFrame([
  {k: v for k, v in overall_all.items() if k != "pivot"},
  {k: v for k, v in overall_no_cola.items() if k != "pivot"},
])
overall_summary["friedman_p_holm"] = holm(overall_summary["friedman_p_raw"]) # perform holm correction on friedman raw results
overall_summary["significant"] = overall_summary["friedman_p_holm"] < 0.05   # check if significant by 0.05

print("Overall top 3 counts")
print("This indicates which top3 expert counts we derive using cola and excluding cola")
display(overall_summary)

print("Now we look closer into the top3 beest experts coutns and check whether the difference between 1st and 2nd best and 2nd abd 3rd best is significant")
print("Wilcoxon: adjacent top counts, with CoLA")
display(wilcoxon_adjacent_top3(overall_all))

print("Wilcoxon: adjacent top counts, without CoLA")
display(wilcoxon_adjacent_top3(overall_no_cola))

# Per benchmark family: best count and significance
print("Now we check per benchmark family which expert count is the best. We only use the best corrected friedman result here. No additional Wilconxon")
print("Per benchmark family, with CoLA")
display(friedman_count_by_benchmark(metrics, "with_cola"))

print("Per benchmark family, without CoLA")
display(friedman_count_by_benchmark(metrics[metrics.kind.ne("cola")], "no_cola"))

Overall top 3 counts
This indicates which top3 expert counts we derive using cola and excluding cola


,analysis,top3_counts,friedman_p_raw,kendalls_w,avg_rank_2,avg_rank_4,avg_rank_8,avg_rank_16,avg_rank_32,friedman_p_holm,significant
0,with_cola,"[2, 4, 8]",3.073507e-18,0.525397,1.738095,2.047619,3.071429,4.547619,3.595238,6.147015e-18,True
1,no_cola,"[32, 2, 4]",8.742769e-15,0.428345,2.547619,2.738095,4.142857,3.928571,1.642857,8.742769e-15,True


Now we look closer into the top3 beest experts coutns and check whether the difference between 1st and 2nd best and 2nd abd 3rd best is significant
Wilcoxon: adjacent top counts, with CoLA


,analysis,a,b,mean_a,mean_b,median_delta_a_minus_b,n_pairs,p_raw,p_holm,significant
0,with_cola,2,4,0.890187,0.886199,0.000778,42,6.564972e-02,0.065650,False
1,with_cola,4,8,0.886199,0.874452,0.005422,42,8.947236e-07,0.000002,True


Wilcoxon: adjacent top counts, without CoLA


,analysis,a,b,mean_a,mean_b,median_delta_a_minus_b,n_pairs,p_raw,p_holm,significant
0,no_cola,32,2,0.903683,0.884799,0.001928,42,0.000002,0.000004,True
1,no_cola,2,4,0.884799,0.882127,0.000865,42,0.083413,0.083413,False


Now we check per benchmark family which expert count is the best. We only use the best corrected friedman result here. No additional Wilconxon
Per benchmark family, with CoLA


,analysis,score_family,winner_count,n_units,p_raw,kendalls_w,avg_rank_2,avg_rank_4,avg_rank_8,avg_rank_16,avg_rank_32,p_holm,significant
0,with_cola,Belebele,2,9,1.110009e-06,0.920988,1.111111,2.555556,4.000000,5.000000,2.333333,0.000003,True
1,with_cola,FLORES en->x,2,9,6.709404e-07,0.950617,1.000000,2.000000,3.000000,4.555556,4.444444,0.000003,True
2,with_cola,FLORES x->en,2,9,6.709404e-07,0.950617,1.000000,2.000000,3.000000,4.555556,4.444444,0.000003,True
3,with_cola,SIB200,4,9,5.431643e-06,0.827160,2.888889,1.000000,2.444444,3.888889,4.777778,0.000011,True
4,with_cola,Taxi1500,32,6,2.681437e-03,0.677778,3.166667,3.000000,2.833333,4.833333,1.166667,0.002681,True


Per benchmark family, without CoLA


,analysis,score_family,winner_count,n_units,p_raw,kendalls_w,avg_rank_2,avg_rank_4,avg_rank_8,avg_rank_16,avg_rank_32,p_holm,significant
0,no_cola,Belebele,32,9,5.915120e-07,0.958025,2.000000,3.111111,4.888889,4.000000,1.000000,0.000003,True
1,no_cola,FLORES en->x,32,9,9.386181e-07,0.930864,1.555556,3.000000,4.111111,4.888889,1.444444,0.000004,True
2,no_cola,FLORES x->en,32,9,9.386181e-07,0.930864,1.777778,3.000000,4.222222,4.777778,1.222222,0.000004,True
3,no_cola,SIB200,4,9,4.079662e-04,0.567901,4.222222,1.222222,3.777778,2.444444,3.333333,0.000816,True
4,no_cola,Taxi1500,32,6,1.638682e-02,0.505556,3.500000,3.666667,3.500000,3.333333,1.000000,0.016387,True


### Interpretation

All results are based on the normalized task-level scores.
For each task/language/metric, we first min-max normalize scores across runs, then aggregate these normalized
scores across benchmark families.

#### Overall expert/component-count ranking

If CoLA is included:

- Top 3 counts are: `2`, `4`, `8`
- The difference between rank 1 and rank 2 (`2` vs `4`) is **not significant**
- This result is likely influenced by CoLA collapsing at higher counts (`16`, `32`)

If CoLA is excluded:

- Top 3 counts are: `32`, `2`, `4`
- The difference between rank 1 and rank 2 (`32` vs `2`) is **significant**
- This suggests that the low-count result is not stable once the CoLA failure mode is removed

#### Per-benchmark Friedman winners

Without CoLA:

| Benchmark family | Best count |
|---|---:|
| Belebele | 32 |
| FLORES en->x | 32 |
| FLORES x->en | 32 |
| SIB200 | 4 |
| Taxi1500 | 32 |

With CoLA:

| Benchmark family | Best count |
|---|---:|
| Belebele | 2 |
| FLORES en->x | 2 |
| FLORES x->en | 2 |
| SIB200 | 4 |
| Taxi1500 | 32 |

# Which specific configurations work best for the top 2 expert counts?

We now use the top 2 expert counts from the no-CoLA analysis and check, for each of those counts separately, which concrete run configuration performs best.


In [24]:
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests
import numpy as np
import pandas as pd

# We now look at the two best expert counts from the no-CoLA analysis.
# For each expert count, we compare all concrete run configs with this count.
# Example concrete configs: moelora_r2_e32_k2, hmora_r2_e32_k4, hydralora_r2_heads32, mtllora_r2_nup32.

def holm(pvals):
    return multipletests(pd.Series(pvals, dtype=float), method="holm")[1]


def concrete_config_pivot(df, group_cols):
    return (
      df.groupby(group_cols + ["run_id"])["normalized_score"]
      .mean()
      .unstack("run_id")
      .dropna()
    )


def best_configs_for_expert_count(df, expert_count):
    selected = df[df.component_count.eq(expert_count)].copy()

    # Descriptive ranking of concrete run configs inside the selected expert count.
    config_summary = (
      selected.groupby(["component_count", "run_id", "kind", "top_k"], dropna=False)
      .agg(
          mean_score=("normalized_score", "mean"),
          n_rows=("normalized_score", "count"),
      )
      .sort_values("mean_score", ascending=False)
      .reset_index()
    )

    candidates = config_summary["run_id"].tolist()
    pivot = concrete_config_pivot(
      selected[selected.run_id.isin(candidates)],
      ["score_family", "task", "language", "metric"],
    )

    # Friedman tests whether the concrete configs for this expert count differ overall.
    stat, p = friedmanchisquare(*[pivot[c] for c in candidates])
    avg_ranks = pivot.rank(axis=1, ascending=False).mean().sort_values() # lower average rank is better
    ordered = avg_ranks.index.tolist()

    friedman_result = pd.DataFrame([{
      "expert_count": expert_count,
      "top_configs_by_friedman_rank": ordered,
      "n_pairs": len(pivot),
      "friedman_p_raw": p,
      "friedman_p_holm": holm([p])[0],
      "significant": holm([p])[0] < 0.05,
      "kendalls_w": stat / (len(pivot) * (pivot.shape[1] - 1)),
      **{f"avg_rank_{c}": avg_ranks.get(c, np.nan) for c in ordered},
    }])

    # Post-hoc: compare adjacent top Friedman-ranked configs: 1st vs 2nd, 2nd vs 3rd, and 3rd vs 4th using Wilcoxon + Holm.
    rows = []
    adjacent_pairs = list(zip(ordered[:3], ordered[1:4]))
    for a, b in adjacent_pairs:
      w = wilcoxon(pivot[a], pivot[b], zero_method="pratt", alternative="greater")
      rows.append({
          "a": a,
          "b": b,
          "mean_a": pivot[a].mean(),
          "mean_b": pivot[b].mean(),
          "median_delta_a_minus_b": (pivot[a] - pivot[b]).median(),
          "n_pairs": len(pivot),
          "p_raw": w.pvalue,
      })

    pairwise = pd.DataFrame(rows)
    pairwise["p_holm"] = holm(pairwise["p_raw"])
    pairwise["significant"] = pairwise["p_holm"] < 0.05
    return config_summary, friedman_result, pairwise


print("Without CoLA: best concrete configs within the top 2 expert counts")
no_cola_metrics = metrics[metrics.kind.ne("cola")].copy()
top_counts_no_cola = overall_no_cola["top3_counts"][:2]
print("Top expert counts:", top_counts_no_cola)

for expert_count in top_counts_no_cola:
  print(f"Expert count {expert_count}: best concrete configs")
  config_summary, config_friedman, config_pairwise = best_configs_for_expert_count(no_cola_metrics, expert_count)

  print("Concrete configs sorted by mean normalized score")
  display(config_summary.head(10))

  print("Friedman test across all concrete configs for this expert count")
  display(config_friedman)

  print("Ordering by Friedman and mean score can be different")
  print("For significance claims we should use Friedman, because it is more consistency-based, less prone to outliers, and therefore statistically more reliable")

  print("Wilcoxon post-hoc: 1st vs 2nd, 2nd vs 3rd, and 3rd vs 4th")
  display(config_pairwise)


Without CoLA: best concrete configs within the top 2 expert counts
Top expert counts: [32, 2]
Expert count 32: best concrete configs
Concrete configs sorted by mean normalized score


,component_count,run_id,kind,top_k,mean_score,n_rows
0,32,moelora_r2_e32_k8,moelora,8.0,0.935823,42
1,32,adamoe_1_r2_e32_k2,mola,2.0,0.930762,42
2,32,adamoe_1_r2_e32_k4,mola,4.0,0.925381,42
3,32,adamoe_1_r2_e32_k16,mola,16.0,0.917686,42
4,32,moelora_r2_e32_k4,moelora,4.0,0.916821,42
5,32,hmora_r2_e32_k2,hmora,2.0,0.907586,42
6,32,adamoe_1_r2_e32_k8,mola,8.0,0.907511,42
7,32,moelora_r2_e32_k2,moelora,2.0,0.906169,42
8,32,hydralora_r2_heads32,hydralora,NaN,0.906068,42
9,32,mov_dense_e32,movlora,NaN,0.900702,42


Friedman test across all concrete configs for this expert count


,expert_count,top_configs_by_friedman_rank,n_pairs,friedman_p_raw,friedman_p_holm,significant,kendalls_w,avg_rank_moelora_r2_e32_k8,avg_rank_adamoe_1_r2_e32_k4,avg_rank_moelora_r2_e32_k4,...,avg_rank_adamoe_1_r2_e32_k1,avg_rank_moelora_r2_e32_k1,avg_rank_moelora_r2_e32_k2,avg_rank_mov_dense_e32,avg_rank_mtllora_r2_nup32,avg_rank_hmora_r2_e32_k16,avg_rank_hmora_r2_e32_k2,avg_rank_hmora_r2_e32_k4,avg_rank_hmora_r2_e32_k8,avg_rank_hmora_r2_e32_k1
0,32,"[moelora_r2_e32_k8, adamoe_1_r2_e32_k4, moelor...",42,2.879303e-47,2.879303e-47,True,0.395597,4.964286,5.440476,5.559524,...,8.47619,8.595238,8.690476,8.904762,9.559524,12.428571,12.571429,13.535714,13.904762,13.964286


Ordering by Friedman and mean score can be different
For significance claims we should use Friedman, because it is more consistency-based, less prone to outliers, and therefore statistically more reliable
Wilcoxon post-hoc: 1st vs 2nd, 2nd vs 3rd, and 3rd vs 4th


,a,b,mean_a,mean_b,median_delta_a_minus_b,n_pairs,p_raw,p_holm,significant
0,moelora_r2_e32_k8,adamoe_1_r2_e32_k4,0.935823,0.925381,0.000155,42,0.063570,0.190710,False
1,adamoe_1_r2_e32_k4,moelora_r2_e32_k4,0.925381,0.916821,0.000046,42,0.150997,0.301994,False
2,moelora_r2_e32_k4,adamoe_1_r2_e32_k2,0.916821,0.930762,0.000027,42,0.601177,0.601177,False


... (2 outputs left out)

Friedman test across all concrete configs for this expert count


,expert_count,top_configs_by_friedman_rank,n_pairs,friedman_p_raw,friedman_p_holm,significant,kendalls_w,avg_rank_hydralora_r2_heads2,avg_rank_mov_dense_e2,avg_rank_adamoe_1_r2_e2_k1,avg_rank_mtllora_r2_nup2,avg_rank_adamole_r2_e2,avg_rank_moelora_r2_e2_k1,avg_rank_hmora_r2_e2_k1
0,2,"[hydralora_r2_heads2, mov_dense_e2, adamoe_1_r...",42,1.301991e-17,1.301991e-17,True,0.364148,2.380952,3.238095,3.297619,3.547619,4.202381,5.107143,6.22619


Ordering by Friedman and mean score can be different
For significance claims we should use Friedman, because it is more consistency-based, less prone to outliers, and therefore statistically more reliable
Wilcoxon post-hoc: 1st vs 2nd, 2nd vs 3rd, and 3rd vs 4th


,a,b,mean_a,mean_b,median_delta_a_minus_b,n_pairs,p_raw,p_holm,significant
0,hydralora_r2_heads2,mov_dense_e2,0.906931,0.900182,0.000369,42,0.500000,1.000000,False
1,mov_dense_e2,adamoe_1_r2_e2_k1,0.900182,0.881197,0.000065,42,0.045703,0.137109,False
2,adamoe_1_r2_e2_k1,mtllora_r2_nup2,0.881197,0.897705,0.000061,42,0.640169,1.000000,False


## Answer to our questions:
### Which is the two best expert configurations:
    - the best rank configurations are `32` and `2` if we exlcude cola since it skews results for higher expert count
    - if we include cola it changes to `2` and `4`
    - I decided here to consider `32` and `2` as the best expert counts since cola clearly skews the resutls for higher amount of expert counts
---
### Which specific approaches perform the best within these best expert counts:
    - the best three approaches when the expert count is `32` are: 1. `moelora_r2_e32_k8` 2. `adamoe_1_r2_e32_k4` 3. `moelora_r2_e32_k4`
    - the best three approaches when the expert count is `2` are: 1. `hydralora_r2_heads2` 2. `mov_dense_e2` 3. `adamoe_1_r2_e2_k1`